Here, I have been testing to see if the issue with the code is formatting or framing (i.e baud rate, parity, start/stop bits, voltage mismatch etc.

The garbled bytes responses from the below cells rule out the possibility of this being a formatting (test by changing what test_cmd is) or framing (second cell cycles different framing paradigms) issue. All thats left is voltage mismatch - which in 99% of cases is the hardware.

In [1]:
import serial
import time

# --- CONFIGURE PORT ---
PORT = "COM4"  # update if needed
BAUD = 9600

# --- CONNECT ---
try:
    ser = serial.Serial(
        port=PORT,
        baudrate=BAUD,
        bytesize=serial.EIGHTBITS,
        parity=serial.PARITY_NONE,
        stopbits=serial.STOPBITS_ONE,
        timeout=1
    )
    print(f"[OK] Connected to {PORT}")
except Exception as e:
    print(f"[ERROR] Could not open port: {e}")
    ser = None





[OK] Connected to COM4


In [5]:
import serial, time

PORT = "COM4"
for baud in [9600, 19200]:
    for parity in [serial.PARITY_NONE, serial.PARITY_EVEN]:
        for stop in [serial.STOPBITS_ONE, serial.STOPBITS_TWO]:
            print(f"\nTesting: {baud} baud, {parity} parity, {stop} stop bits")
            try:
                ser = serial.Serial(port=PORT, baudrate=baud, bytesize=8,
                                    parity=parity, stopbits=stop, timeout=1)
                ser.write(b"@00VER\r")
                time.sleep(0.2)
                reply = ser.read(64)
                print("Raw:", reply, "Hex:", reply.hex())
                ser.close()
            except Exception as e:
                print("Error:", e)



Testing: 9600 baud, N parity, 1 stop bits
Raw: b'\xff\xff' Hex: ffff

Testing: 9600 baud, N parity, 2 stop bits
Raw: b'\xff' Hex: ff

Testing: 9600 baud, E parity, 1 stop bits
Raw: b'\xff' Hex: ff

Testing: 9600 baud, E parity, 2 stop bits
Raw: b'\xff\xff' Hex: ffff

Testing: 19200 baud, N parity, 1 stop bits
Raw: b'\xff\xff\xff' Hex: ffffff

Testing: 19200 baud, N parity, 2 stop bits
Raw: b'\xff\xff\xff\xff' Hex: ffffffff

Testing: 19200 baud, E parity, 1 stop bits
Raw: b'\xff\xff\xff' Hex: ffffff

Testing: 19200 baud, E parity, 2 stop bits
Raw: b'\xff\xff\xff\xff' Hex: ffffffff


In [1]:
import serial
import time
from Pump import AL1000

ser = serial.Serial("COM4", baudrate=9600, parity=serial.PARITY_NONE,
                    stopbits=serial.STOPBITS_ONE, bytesize=8, timeout=2,
                    rtscts=False, dsrdtr=False)

pump = AL1000(ser)
pump.connect()
response = pump.identify()
print("Pump identify response:", response)


Device is connected
Sent: @00ID | Reply: �
Pump identify response: �


In [2]:

ser.write(bytes([0x02, 0x08]) + b"SAF0UC" + bytes([0x03]))
time.sleep(0.5)  # give pump time to process


In [3]:
reply = ser.read(64)
print("Response after switching to basic mode:", reply)

Response after switching to basic mode: b'\x7f\xff'


In [4]:
response = pump.identify()
print("Pump identify response:", response)

Sent: @00ID | Reply: ��
Pump identify response: ��


In [2]:
import serial.tools.list_ports

for port in serial.tools.list_ports.comports():
    print(port.device, port.name, port.description)


COM3 COM3 Intel(R) Active Management Technology - SOL (COM3)
COM1 COM1 Communications Port (COM1)
COM4 COM4 Prolific PL2303GT USB Serial COM Port (COM4)


In [3]:
import serial
import time

ser = serial.Serial("COM4", 9600, bytesize=8, parity='N', stopbits=1, timeout=2)


In [4]:
# Send the BASIC mode packet
ser.write(b"\x02\x08SAF0UC\x03")
time.sleep(0.5)  # wait for the pump to process
response = ser.read(50)
print("Response after switching to BASIC mode:", response)


Response after switching to BASIC mode: b'\x81\xae\xd3\xf5\x1a'


In [1]:
import serial
ser = serial.Serial("COM4", 9600, timeout=1)
print("COM4 opened successfully")
ser.close()


COM4 opened successfully


In [2]:
from testing import format_command
import serial, time

ser = serial.Serial("COM4", 9600, timeout=1)
cmd = "SAF0"  # turn safe mode off
formatted = format_command(0, cmd)
ser.write(formatted.encode())
time.sleep(0.2)
resp = ser.read(50)
print("Raw safe-mode response:", resp)
ser.close()


Raw safe-mode response: b'\xff'


In [3]:
from testing import interpret_response

try:
    address, status, msg = interpret_response(resp.decode(), basic=False)
    print("Parsed response:", address, status, msg)
except Exception as e:
    print("Error parsing response:", e)


Error parsing response: 'utf-8' codec can't decode byte 0xff in position 0: invalid start byte
